# MODIS discovery — Guanajuato (earthaccess + pyhdf)

This notebook is a **scientific smoke test**, not an output certified for modeling.

**Location:** 20.4243195, -101.1466639 (regional reference, not a surveyed plot).

The previous FarmAdapt V2 used **ORNL REST JSON** for MODIS; its extraction did not require reading HDF4. Here we retain NASA CMR acquisition with **earthaccess** and read **native LP DAAC HDF4** directly using **pyhdf**, without GDAL's optional HDF4 plugin.

From the repository root install: `python -m pip install -e '.[source,source-test,source-modis]'`. Earthdata login is required only for live downloads.

**No network requests are made unless you set RUN_LIVE=True.**

In [8]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from gradientdescentfarmers.source_builder.adapter import (
    REPO_ROOT, load_source_config, NASAProfileSourceBuilder
)
from gradientdescentfarmers.source_builder.core import modis_earthaccess as modis
from gradientdescentfarmers.source_builder.core.preprocessing import (
    standardize_mod13, standardize_mod11, standardize_mod16
)
from gradientdescentfarmers.source_builder.core.validation import validate_table

LAT, LON = 20.4243195, -101.1466639
print("Repo:", REPO_ROOT)
print("Reference:", LAT, LON)

Repo: /Users/danielmalvaez/Hackathons/GradientDescentFarmers
Reference: 20.4243195 -101.1466639


In [ ]:
# Preflight: verify the notebook kernel and credential locations (never print secrets).
import os, sys, importlib.util
from gradientdescentfarmers.source_builder.core.auth import load_project_env
found_envs = load_project_env()
print('Python executable:', sys.executable)
print('earthaccess installed:', importlib.util.find_spec('earthaccess') is not None)
print('Detected .env locations:', [str(p) for p in found_envs])
print('Earthdata credentials present:', bool(os.getenv('EARTHDATA_TOKEN')) or
      bool(os.getenv('EARTHDATA_USERNAME') and os.getenv('EARTHDATA_PASSWORD')))
assert importlib.util.find_spec('earthaccess') is not None, 'Install the source extras in this kernel'


## 1 — Native reader and sinusoidal coordinate test

The complete MODLAND sinusoidal grid uses radius 6371007.181 m. We verify the h/v tile and the containing row/column before accessing an SDS. HDF4 support in rasterio is **not** required.

In [9]:
import pyhdf
import pyproj
print("pyhdf:", pyhdf.__file__)
print("pyproj:", pyproj.__version__)
x, y = modis._sinusoidal_xy(LAT, LON)
h = int(np.floor((x - modis.X_MIN_M)/modis.TILE_M))
v = int(np.floor((modis.Y_MAX_M - y)/modis.TILE_M))
tile = f"h{h:02d}v{v:02d}"
test_file = Path(f"MOD13Q1.A2024185.{tile}.061.TEST.hdf")
print("Sinusoidal xy (m):", x, y, "Tile:", tile)
for n in (1200, 2400, 4800):
    print("SDS", n, modis.pixel_for_point(test_file, LAT, LON, (n, n)))

pyhdf: /Users/danielmalvaez/miniconda3/envs/Kernelsote/lib/python3.11/site-packages/pyhdf/__init__.py
pyproj: 3.6.1
Sinusoidal xy (m): -10539953.567987435 2271083.268390253 Tile: h08v06
SDS 1200 (1149, 625)
SDS 2400 (2298, 1250)
SDS 4800 (4596, 2501)


## 2 — Inspect existing cached MODIS HDF4 granules (no download)

Looks only under the repository's ignored `data/cache/earthdata/modis/` folder. If nothing is there, proceed to section 3 and explicitly enable the pilot.

In [10]:
cache = REPO_ROOT / "data/cache/earthdata/modis"
products = ("MOD13Q1", "MOD11A2", "MOD16A2GF")
cached = {product: sorted((cache/product).glob("**/*.hdf")) for product in products}
for product, paths in cached.items():
    print(product, len(paths), "files")
    for path in paths[:2]:
        print(" ", path.relative_to(REPO_ROOT), f"{path.stat().st_size:,} bytes")

MOD13Q1 0 files
MOD11A2 0 files
MOD16A2GF 0 files


## 3 — Live pilot (requires explicit opt-in)

**Start with one MOD13Q1 granule**, not the entire 2024–2025 series. This may download a large regional HDF4 tile. It does **not** overwrite the older `data/processed/` or GPM/SMAP runs.

Set `RUN_LIVE = True` only when ready. Then inspect `availability.json` if a source fails. For all 3 products set `RUN_ALL_PRODUCTS = True` after MOD13Q1 succeeds.

In [11]:
RUN_LIVE = True
RUN_ALL_PRODUCTS = True
new_run = None

if RUN_LIVE:
    config = load_source_config(REPO_ROOT/"resources/guanajuato_modis_earthaccess.json")
    settings = json.loads((REPO_ROOT/"resources/guanajuato_modis_earthaccess_settings.json").read_text())
    settings["modis_max_granules"] = 1
    for source in ("modis_mod13q1", "modis_mod11a2", "modis_mod16a2gf"):
        settings["source_overrides"][source] = (
            RUN_ALL_PRODUCTS or source == "modis_mod13q1"
        )
    config.pop("settings_file", None)
    config["settings"] = settings
    config["output_root"] = "data/modis_discovery"
    config["run"] = True
    new_run = NASAProfileSourceBuilder().build(config)
    print("Run:", new_run["run_id"])
    print("Status:", new_run["source_status"])
    print("Output:", new_run["output_dir"])
else:
    print("No downloads attempted. Set RUN_LIVE=True to run the pilot.")

/Users/danielmalvaez/miniconda3/envs/Kernelsote/lib/python3.11/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/Users/danielmalvaez/miniconda3/envs/Kernelsote/lib/python3.11/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


Run: 38b7c3db5ba44063
Status: {'nasa_power': 'not_requested', 'srtm': 'not_requested', 'soilgrids_wcs': 'not_requested', 'modis_mod13q1': 'partial', 'modis_mod11a2': 'partial', 'modis_mod16a2gf': 'partial', 'gpm_imerg_v07': 'not_requested', 'smap_l4': 'not_requested', 'worldbank_agriculture': 'not_requested', 'usda_quickstats': 'not_requested', 'faostat_production': 'not_requested', 'faostat_prices': 'not_requested', 'grace_fo_mascon': 'not_requested', 'nex_gddp_cmip6': 'not_requested', 'modis_landcover': 'not_requested', 'farmer_crop_history': 'not_requested', 'farmer_soil_tests': 'not_requested', 'validated_crop_parameters': 'missing', 'local_crop_prices': 'missing', 'local_soil_lab': 'not_requested', 'management_history': 'not_requested'}
Output: /Users/danielmalvaez/Hackathons/GradientDescentFarmers/data/modis_discovery/guanajuato_valle_demo/runs/38b7c3db5ba44063


## 4 — Inspect SDS names, shapes, raw pixel, and quality flags

Uses **pyhdf** only. A native MODIS file has scientific datasets (SDS), including quality layers. Missing SDS or mismatched tile geometry is reported as an error; no values are invented.

Do not judge science quality by download success alone. QC may reject cloudy or unreliable pixels.

In [12]:
cached = {product: sorted((cache/product).glob("**/*.hdf")) for product in products}
raw_points = {}
for product, paths in cached.items():
    if not paths:
        print(product, ": no HDF4 yet")
        continue
    path = paths[0]
    print("
PRODUCT", product, "FILE", path.name)
    datasets = modis.list_science_datasets(path)
    print("SDS:", list(datasets)[:25])
    required = modis.LAYERS[product]
    missing = sorted(set(required.values()) - set(datasets))
    assert not missing, f"Missing native SDS for {product}: {missing}"
    result = modis.extract_point(path, LAT, LON, product)
    raw_points[product] = result
    print(pd.Series({k:v for k,v in result.items() if not k.endswith(("_row","_col","_shape"))}))


PRODUCT MOD13Q1 FILE MOD13Q1.A2023353.h08v06.061.2024005164941.hdf
SDS: ['250m 16 days NDVI', '250m 16 days EVI', '250m 16 days VI Quality', '250m 16 days red reflectance', '250m 16 days NIR reflectance', '250m 16 days blue reflectance', '250m 16 days MIR reflectance', '250m 16 days view zenith angle', '250m 16 days sun zenith angle', '250m 16 days relative azimuth angle', '250m 16 days composite day of the year', '250m 16 days pixel reliability']
date                                                  2023-12-19 00:00:00+00:00
modis_date                                                             A2023353
source_file                       MOD13Q1.A2023353.h08v06.061.2024005164941.hdf
tile                                                                     h08v06
250m_16_days_NDVI                                                        4673.0
250m_16_days_EVI                                                         1982.0
250m_16_days_VI_Quality                                            

## 5 — Apply *original* FarmAdapt transformations and QA checks

The same functions are used for both ORNL and HDF4 pathways; the HDF4 reader supplies raw integers and leaves QA bits unmodified.

In [13]:
transformers = {
    "MOD13Q1": ("vegetation_timeseries", standardize_mod13),
    "MOD11A2": ("land_surface_temperature", standardize_mod11),
    "MOD16A2GF": ("evapotranspiration", standardize_mod16),
}
for product, record in raw_points.items():
    name, transformer = transformers[product]
    frame = pd.DataFrame([{**record, "farm_id": "guanajuato_valle_demo"}])
    standardized = transformer(frame)
    report = validate_table(name, standardized).as_dict()
    print(product, "| QA passed:", report["passed"],
          "| usable rows:", report["usable_rows"], "| coverage:", report["coverage"])
    print(standardized.head(1).to_string(index=False)[:1500])
    if not report["passed"]:
        print("QA checks:", [c for c in report["checks"] if not c["passed"]])

MOD13Q1 | QA passed: True | usable rows: 1 | coverage: 1.0
                     date modis_date               farm_id  ndvi_raw  evi_raw  vi_quality  pixel_reliability   ndvi    evi  mandatory_qa  vi_usefulness  aerosol  mixed_cloud  possible_shadow  qa_pass  ndvi_qa  evi_qa
2023-12-19 00:00:00+00:00   A2023353 guanajuato_valle_demo    4673.0   1982.0      2116.0                0.0 0.4673 0.1982             0              1        1            0                0     True   0.4673  0.1982
MOD11A2 | QA passed: True | usable rows: 1 | coverage: 1.0
                     date modis_date               farm_id  lst_day_raw  qc_day  lst_day_c  day_mandatory_qa  day_lst_error_class  day_qa_pass  lst_day_qa_c  lst_night_raw  qc_night  lst_night_c  night_mandatory_qa  night_lst_error_class  night_qa_pass  lst_night_qa_c
2024-01-01 00:00:00+00:00   A2024001 guanajuato_valle_demo      15044.0     0.0      27.73                 0                    0         True         27.73        14089.0       0

## 6 — Inspect source errors without another network request

Review the last pilot's availability metadata and the previously failed GPM/SMAP smoke run. The top-level `build_state=failed` is not enough to diagnose authentication, download, SDS, or extraction errors.

In [14]:
candidate_runs = []
if new_run:
    candidate_runs.append(Path(new_run["output_dir"]))
candidate_runs.append(
    REPO_ROOT/"data/guanajuato_smoke/guanajuato_valle_demo/runs/5508cf72d72d40f4"
)
for run in candidate_runs:
    path = run/"availability.json"
    if not path.is_file():
        print("Not found:", path)
        continue
    statuses = json.loads(path.read_text())
    print("
RUN:", run.name)
    for source in ("modis_mod13q1", "modis_mod11a2", "modis_mod16a2gf",
                   "gpm_imerg_v07", "smap_l4"):
        rec = statuses.get(source, {})
        if rec.get("status") in ("error", "partial"):
            print(source, rec.get("status"), rec.get("error"), rec.get("warnings"))


RUN: 38b7c3db5ba44063
modis_mod13q1 partial None ['CMR query reached the requested max granules; completeness not established']
modis_mod11a2 partial None ['CMR query reached the requested max granules; completeness not established']
modis_mod16a2gf partial None ['CMR query reached the requested max granules; completeness not established']

RUN: 5508cf72d72d40f4
gpm_imerg_v07 error ModuleNotFoundError: No module named 'earthaccess' []
smap_l4 error ModuleNotFoundError: No module named 'earthaccess' []


## 7 — Decision gate

Proceed to the 2-granule 3-product run only if: download works, pyhdf lists expected SDS, the h/v tile agrees with Guanajuato, native values are extracted and the QA evaluation is auditable.

The notebook itself is exploratory. **One or two granules are not full MODIS temporal coverage.**